# Instructions
You need to run the below bash commands through the terminal of lightning.ai. 
Remember to create a new studio before running any commands. 


# Pseudo-Distributed Hadoop Cluster Setup on Ubuntu (SSH Port 2222)

---

## **Prerequisites**
- Ubuntu (20.04/22.04)
- Java 11
- SSH Server

---

## **1. Install Java**
```bash
sudo apt update
sudo apt install openjdk-11-jdk -y
java -version  # Verify installation
```

## **2. Create Hadoop User and Group**
```bash
sudo groupadd hadoop
sudo useradd -m -d /home/hadoop -s /bin/bash -g hadoop hadoop
sudo usermod -aG sudo hadoop
sudo passwd hadoop  # Set a password for the hadoop user
su - hadoop
```

## **3. 3. Configure SSH Server on Port 2222**

### 3.1. Install SSH Server
```bash
sudo apt install openssh-server openssh-client -y
```

### 3.2. Edit SSH Configuration
```bash
sudo sed -i 's/^#Port 22/Port 2222/' /etc/ssh/sshd_config
sudo sed -i 's/^#PermitRootLogin prohibit-password/PermitRootLogin no/' /etc/ssh/sshd_config
sudo sed -i 's/^#PasswordAuthentication yes/PasswordAuthentication no/' /etc/ssh/sshd_config
sudo sed -i 's/^#PubkeyAuthentication yes/PubkeyAuthentication yes/' /etc/ssh/sshd_config
```
### 3.3 Restart SSH
```bash
sudo systemctl restart ssh
```

### 3.4 Verify SSH Port
```bash
sudo ss -tulnp | grep sshd
```

## 4. Set up SSH Passwordless login

### 4.1 Generate SSH key
```bash
ssh-keygen -t rsa -P '' -f ~/.ssh/id_rsa
```

### 4.2 Add Public keys to authorized keys
```bash
cat ~/.ssh/id_rsa.pub >> ~/.ssh/authorized_keys
chmod 600 ~/.ssh/authorized_keys
```

### 4.3 Configure SSH client for Port 2222
```bash
cat > ~/.ssh/config <<EOF
Host localhost
    HostName localhost
    Port 2222
    User hadoop
    IdentityFile ~/.ssh/id_rsa
EOF
```

### 4.4 Test connection
```bash
ssh -p 2222 localhost
```

## 5. Set environment variables

### 5.1 Set System-Wide JAVA_HOME
```bash
echo "JAVA_HOME=\"/usr/lib/jvm/java-11-openjdk-amd64\"" | sudo tee -a /etc/environment
source /etc/environment
```

### 5.2 Set JAVA_HOME for Hadoop User
```bash 
echo 'export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64' >> ~/.bashrc
echo 'export HADOOP_HOME=/usr/local/hadoop' >> ~/.bashrc
#echo 'export PATH=\$PATH:\$JAVA_HOME/bin:\$HADOOP_HOME/bin:\$HADOOP_HOME/sbin' >> ~/.bashrc
echo 'export PATH="$PATH:$JAVA_HOME/bin:$HADOOP_HOME/bin:$HADOOP_HOME/sbin"' >> ~/.bashrc
source ~/.bashrc
```


## 6. Download and Install Hadoop
```bash
exit  # Temporarily return to original user for installation
sudo wget https://downloads.apache.org/hadoop/common/hadoop-3.3.6/hadoop-3.3.6.tar.gz -P /tmp
sudo tar -xzf /tmp/hadoop-3.3.6.tar.gz -C /usr/local
sudo mv /usr/local/hadoop-3.3.6 /usr/local/hadoop
sudo chown -R hadoop\:hadoop /usr/local/hadoop
su - hadoop  # Switch back to hadoop user
```

## 7. Configure Hadoop

### 7.1 Update hadoop-env.sh
```bash

echo 'export JAVA_HOME=/usr/lib/jvm/java-11-openjdk-amd64' >> $HADOOP_HOME/etc/hadoop/hadoop-env.sh
echo 'export HADOOP_SSH_OPTS="-p 2222"' >> $HADOOP_HOME/etc/hadoop/hadoop-env.sh


```


### 7.2. Edit Hadoop Configuration Files Using sed

#### core-site.xml

```bash
sudo sed -i '/<\/configuration>/i \
  <property>\
    <name>fs.defaultFS<\/name>\
    <value>hdfs:\/\/localhost:9000<\/value>\
  <\/property>' \/usr/local/hadoop//etc/hadoop/core-site.xml
```
#### hdfs-site.xml

```bash
sudo sed -i '/<\/configuration>/i \
  <property>\
    <name>dfs.replication<\/name>\
    <value>1<\/value>\
  <\/property>\
  <property>\
    <name>dfs.namenode.name.dir<\/name>\
    <value>file:\/\/\/usr\/local\/hadoop\/data\/namenode<\/value>\
  <\/property>\
  <property>\
    <name>dfs.datanode.data.dir<\/name>\
    <value>file:\/\/\/usr\/local\/hadoop\/data\/datanode<\/value>\
  <\/property>' $HADOOP_HOME/etc/hadoop/hdfs-site.xml
```

#### mapred-site.xml

```bash
sudo sed -i '/<\/configuration>/i \
  <property>\
    <name>mapreduce.framework.name<\/name>\
    <value>yarn<\/value>\
  <\/property>\
  <property>\
    <name>yarn.app.mapreduce.am.env<\/name>\
    <value>HADOOP_MAPRED_HOME=\/usr\/local\/hadoop<\/value>\
  <\/property>\
  <property>\
    <name>mapreduce.map.env<\/name>\
    <value>HADOOP_MAPRED_HOME=\/usr\/local\/hadoop<\/value>\
  <\/property>\
  <property>\
    <name>mapreduce.reduce.env<\/name>\
    <value>HADOOP_MAPRED_HOME=\/usr\/local\/hadoop<\/value>\
  <\/property>' $HADOOP_HOME/etc/hadoop/mapred-site.xml
```

#### yarn-site.xml

```bash
sudo sed -i '/<\/configuration>/i \
  <property>\
    <name>yarn.nodemanager.aux-services<\/name>\
    <value>mapreduce_shuffle<\/value>\
  <\/property>\
  <property>\
    <name>yarn.nodemanager.aux-services.mapreduce.shuffle.class<\/name>\
    <value>org.apache.hadoop.mapred.ShuffleHandler<\/value>\
  <\/property>' $HADOOP_HOME/etc/hadoop/yarn-site.xml
```


### Start Hadoop
```bash 
$HADOOP_HOME/bin/hdfs namenode -format

$HADOOP_HOME/sbin/start-dfs.sh
$HADOOP_HOME/sbin/start-yarn.sh

jps
```

_______

## HDFS Commands

## HDFS `hdfs dfs` Cheat Sheet (with Bash analogues)

> `hdfs dfs` is the HDFS/FS shell. It mirrors many Unix file ops but talks to HDFS (or whatever FS Hadoop is configured to use). `hadoop fs` is effectively a synonym when HDFS is the default filesystem. 
---

### Most-used commands (HDFS ↔︎ Bash)

| HDFS command              | Example                                | What it does                                        | Bash analogue           |             |
| ------------------------- | -------------------------------------- | --------------------------------------------------- | ----------------------- | ----------- |
| `-ls`                     | `hdfs dfs -ls -h -R /data`             | List files/dirs (`-h` human sizes, `-R` recursive). | `ls -lhR`               |             |
| `-du`                     | `hdfs dfs -du -h /data`                | Disk usage per path.                                | `du -h`                 |             |
| `-dus`                    | `hdfs dfs -dus /data/friends`          | Summary size for a path.                            | `du -sh`                |             |
| `-count`                  | `hdfs dfs -count -q /data/friends`     | Count dirs/files/bytes (quota info with `-q`).      | `find …                 | wc -l`+`du` |
| `-mkdir`                  | `hdfs dfs -mkdir -p /data/friends/raw` | Create directory (parents with `-p`).               | `mkdir -p`              |             |
| `-put` / `-copyFromLocal` | `hdfs dfs -put local.txt /data/`       | Copy **local → HDFS**.                              | `cp local.txt /mnt/...` |             |
| `-get` / `-copyToLocal`   | `hdfs dfs -get /data/file .`           | Copy **HDFS → local**.                              | `cp /mnt/... .`         |             |
| `-cp`                     | `hdfs dfs -cp /data/a /data/b`         | Copy **within HDFS**.                               | `cp`                    |             |
| `-mv`                     | `hdfs dfs -mv /data/a /data/b`         | Move/rename **within HDFS**.                        | `mv`                    |             |
| `-rm`                     | `hdfs dfs -rm /data/file`              | Remove file.                                        | `rm`                    |             |
| `-rm -r`                  | `hdfs dfs -rm -r /data/dir`            | Recursive delete. Add `-skipTrash` to bypass Trash. | `rm -rf`                |             |
| `-cat`                    | `hdfs dfs -cat /data/file              | head`                                               | Print file to stdout.   | `cat`       |
| `-text`                   | `hdfs dfs -text /data/file.gz`         | Decode seqfile/avro/parquet/text (auto) to text.    | `zcat`/tool-specific    |             |
| `-tail`                   | `hdfs dfs -tail -f /data/file`         | Print last 1KB; `-f` to follow.                     | `tail -f`               |             |
| `-stat`                   | `hdfs dfs -stat "%n %b %o" /data/file` | Print name/size/replication, etc.                   | `stat`                  |             |
| `-setrep`                 | `hdfs dfs -setrep -w 2 /data/file`     | Set/block until replication = 2.                    | — (HDFS-specific)       |             |
| `-chmod`                  | `hdfs dfs -chmod -R 750 /data/dir`     | Change perms (supports symbolic/octal).             | `chmod -R 750`          |             |
| `-chown`                  | `hdfs dfs -chown user:group /data/dir` | Change owner:group.                                 | `chown user:group`      |             |
| `-chgrp`                  | `hdfs dfs -chgrp -R analytics /data`   | Change group.                                       | `chgrp -R`              |             |
| `-help`                   | `hdfs dfs -help`                       | List all FS shell commands.                         | `man`, `--help`         |             |


## Exercise 1 - Get the Friends text file, upload to HDFS, and preview

Goal: Download the friends transcript we have been using from the github, put it in the hdfs dir /data, and verify contents.


## Exercise 2 — WordCount + Top-K most frequent words

Goal: Run the built-in wordcount job over your text file, then find the top 20 words.



## Exercise 3 (Friends transcripts): speaking turns per character

Goal: count speaking turns by detecting Name: at line start; output only the name via a capture group 


## Exercise 4 - Grep: count dialogues between two characters

Goal: Among your Friends transcript, count lines where two specific characters both appear in the same line, e.g. Monica and Rachel. Use grep with a regex that ensures both names appear (in any order) in the same line. 



## Exercise 5 - Grep: count questions vs exclamations

Goal: Among your transcript file, count how many lines contain a question mark ? (dialogue questions) vs exclamation mark ! (exclamatory lines).


